# OAKG — End-to-End Walkthrough (interactive tutorial)

**Observability-Aware Knowledge Graph Reasoning for Heterogeneous Medical Imaging**

This notebook walks the whole pipeline **from raw segmentation masks to the final
retrieval result**, one step at a time, and calls out the small design decisions
that add up to our novel contribution. Run it top-to-bottom in a meeting — every
cell executes on a small reproducible demo (no external data needed).

> **How to read it:** each section = one stage of the pipeline. Green *Why it
> matters* notes flag the choices that make OAKG novel. The final numbers cited
> are from the real 512-case study (see `results/`); the live cells use demo data.


## 0. The problem in one sentence

We want to **retrieve similar patients** from a database where every case was
imaged/annotated differently — one scan shows only the liver, another only the
pancreas, another all five organs. **How do you compare cases that don't even
observe the same anatomy?**

The naive fix — fill the missing values with zeros and compute cosine similarity —
**hallucinates**: it treats *unobserved* as *absent*, so two unrelated cases can
look similar just because they share a lot of imputed zeros.

**OAKG's idea:** only compare evidence that was *actually observed and
anatomically supported* in both cases, and be explicit about how much shared
evidence there is. That's *observability-awareness*.


In [1]:
import sys, pathlib, json, numpy as np, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import oakg
from oakg import Config
from oakg.data import generate_demo_data, Corpus, validate_data
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 20)
cfg = Config(use_demo_data=True, n_demo_cases=60, seed=2027)
data = generate_demo_data(cfg); validate_data(data)
print('demo benchmark:', len(data.cases), 'cases,', len(data.queries), 'queries')
print('oakg version', oakg.__version__)


demo benchmark: 60 cases, 15 queries
oakg version 0.1.0


## 1. Input: heterogeneous cases (the observability structure)

Each **case** comes from a source dataset that only annotates certain organs.
`available_organs` is the case's *observation set* $O(s)$ — the anatomy we can
actually see. This heterogeneity is the whole point.


In [2]:
# Each case observes a different organ subset (its observation set O(s)).
display(data.cases.groupby('dataset')['available_organs'].first().to_frame('observes'))
data.cases.head(6)


,observes
dataset,
FLARE,left_kidney|liver|pancreas|right_kidney|spleen
LiTS,liver
Pancreas,pancreas


,case_id,dataset,split,available_organs
0,C0000,Pancreas,train,pancreas
1,C0001,LiTS,test,liver
2,C0002,Pancreas,val,pancreas
3,C0003,LiTS,train,liver
4,C0004,LiTS,test,liver
5,C0005,FLARE,val,left_kidney|liver|pancreas|right_kidney|spleen


**Why it matters.** In the real study these are *Pancreas* (pancreas-only),
*LiTS* (liver-only), and *FLARE* (all 5 organs). FLARE is the **multi-organ hub**:
because it shares an organ with everyone, it *bridges* the single-organ datasets —
the connective tissue that makes cross-dataset comparison possible at all.


## 2. From masks to phenotypes (what a 'feature' is)

Upstream, each organ/tumor mask becomes a handful of **phenotypes**: organ
present? organ volume? tumor present? tumor burden? These are the comparable
features. Crucially, every feature carries a **support** — the organ whose
annotation it depends on. `pancreas_tumor_present` is *supported by* the pancreas;
if the pancreas wasn't observed, that feature is genuinely **unobservable**, not
zero.


In [3]:
# The MMKG schema: each feature -> its type and its supporting organ(s).
corpus = Corpus.build(data, cfg)
schema = corpus.schema.copy()
schema['support'] = schema['feature'].map(lambda f: '|'.join(sorted(corpus.feature_support[f])) or '(global)')
schema[['feature','feature_type','support']]


,feature,feature_type,support
0,cross_organ_distribution,binary,left_kidney|liver|pancreas|right_kidney|spleen
1,kidney_tumor_present,binary,left_kidney|right_kidney
2,lesion_multiplicity,numeric,(global)
3,liver_containment,binary,liver
4,liver_tumor_present,binary,liver
5,pancreas_containment,binary,pancreas
6,pancreas_tumor_present,binary,pancreas
7,tumor_burden,numeric,(global)


**Why it matters (the MMKG).** The ontology/schema is a *shared vocabulary*: a
`pancreas` in FLARE is the same concept as a `pancreas` in the Pancreas dataset.
That's what lets heterogeneous sources live in **one graph**. `support` is the
hook OAKG uses to know *which* missing values are truly unobservable.


## 3. The feature matrix — missingness is the signal

Stack the phenotypes into a matrix `X` (cases × features). `M` marks what's
observed. **The `NaN` pattern is not noise — it *is* the observability structure.**


In [4]:
X, M = corpus.x_ref_full, corpus.m_ref_full
obs_frac = M.mean()
print(f'observed fraction of the matrix: {obs_frac:.2f}  (the rest is genuinely unobserved)')
# show one pancreas case and one liver-bearing case: note the complementary NaNs
import numpy as np
frame = pd.DataFrame(X, columns=corpus.features, index=corpus.case_order)
frame.insert(0, 'dataset', data.cases.set_index('case_id')['dataset'])
frame.head(4).T


observed fraction of the matrix: 1.00  (the rest is genuinely unobserved)


,C0000,C0001,C0002,C0003
dataset,Pancreas,LiTS,Pancreas,LiTS
cross_organ_distribution,0.0,0.0,0.0,0.0
kidney_tumor_present,0.0,0.0,0.0,0.0
lesion_multiplicity,3.0,1.0,1.0,1.0
liver_containment,0.0,0.0,0.0,0.0
liver_tumor_present,0.0,0.0,0.0,0.0
pancreas_containment,1.0,0.0,0.0,0.0
pancreas_tumor_present,1.0,0.0,0.0,0.0
tumor_burden,53.214682,17.538017,11.620942,29.366482


## 4. Masking regimes — simulating missing coverage

To stress-test retrieval we *mask* organs at different rates (uniform / random /
dataset-style / asymmetric). Each case gets **its own** observation mask.


In [5]:
from oakg.masking import make_random_mask, apply_mask
real = make_random_mask(data.cases, missing_fraction=0.4, seed=99)
Xm, Mm = apply_mask(corpus.x_ref_full, corpus.m_ref_full, real, corpus)
print('after 40% random masking, observed fraction:', round(Mm.mean(),2))
# a case's retained organ set shrinks:
ex = data.cases.case_id.iloc[0]
print(ex, 'retained organs:', sorted(real.observation_sets[ex]))


after 40% random masking, observed fraction: 0.54
C0000 retained organs: ['pancreas']


## 5. The core: how OAKG scores a pair vs how imputation does

This is the heart of the contribution. Take one **query** and compare it to a
**relevant** case and a **distractor**, three ways. Watch where OAKG differs.


In [6]:
from oakg.baselines import zero_imputed_similarity, masked_cosine_similarity
from oakg.oakg import oakg_similarity_components, observation_overlap, rank_policy

q = data.queries.query_case_id.iloc[0]; qi = corpus.case_to_row[q]
# pick two candidates: nearest-by-observed vs a broad one
cands = [c for c in corpus.case_order if c != q][:2]
for c in cands:
    ci = corpus.case_to_row[c]
    inter, gamma = observation_overlap(q, c, real)   # shared observed organs + coefficient
    s, comps = oakg_similarity_components(qi, ci, Xm, Mm, corpus)
    zero = zero_imputed_similarity(qi, np.array([ci]), Xm, Mm, corpus)[0]
    mask = masked_cosine_similarity(qi, np.array([ci]), Xm, Mm, corpus)[0]
    print(f'{q} vs {c}: shared_organs={sorted(inter)} gamma={gamma:.2f}')
    print(f'   OAKG support-restricted similarity S={s}  (components={comps})')
    print(f'   OAKG-product score = gamma*S = {None if s is None else round(gamma*s,3)}')
    print(f'   zero-imputation cosine = {zero:.3f}   masked cosine = {mask}')
    print()


C0045 vs C0000: shared_organs=['pancreas'] gamma=0.33
   OAKG support-restricted similarity S=0.8556573007942434  (components={'numeric': 0.7113146015884868, 'categorical_relational': 1.0})
   OAKG-product score = gamma*S = 0.285
   zero-imputation cosine = 0.999   masked cosine = 0.9994953026789755

C0045 vs C0001: shared_organs=[] gamma=0.00
   OAKG support-restricted similarity S=0.883081088368485  (components={'numeric': 0.883081088368485})
   OAKG-product score = gamma*S = 0.0
   zero-imputation cosine = 0.999   masked cosine = 0.9995341006857965



**Read the numbers above.** Three levers make OAKG different:
1. **Support-restriction** — it only compares features observed *and* supported in
   both cases (no imputed zeros sneaking in).
2. **$\gamma$ (shared-evidence coefficient)** — Jaccard overlap of the two
   observation sets. Low $\gamma$ = little common ground.
3. **Ranking policy** — how $\gamma$ and similarity $S$ combine:
   `similarity` ($R{=}S$), `product` ($R{=}\gamma S$), `threshold`, `lexicographic`.

> **Why it matters.** Zero-imputation's score is driven by *all* dimensions
> including the imputed ones — that's how it hallucinates. OAKG's score can only
> come from real shared evidence.


## 6. Queries, relevance, and evaluation

A **query** is a structured request (predicates on phenotypes). **Relevance** is
defined from complete reference annotations. We rank candidates and score with
**nDCG@10**, then compare methods with a **paired bootstrap** confidence interval.

**A key design choice: organ-consistent relevance.** A candidate can only be
relevant if it shares an annotated organ with the query — so a coverage-blind
method's cross-organ 'matches' count as *false positives*, not hits.


In [7]:
from oakg.benchmark import run_full_benchmark, summarize
from oakg.masking import build_default_realizations
res = run_full_benchmark(build_default_realizations(data.cases, corpus, cfg), data, corpus, cfg)
agg = summarize(res)
sub = agg[(agg.track=='ref') & (agg.masking_regime=='random')]
sub.sort_values('nDCG@10', ascending=False)[['method','P@10','nDCG@10','ServedRate']].head(10)


,method,P@10,nDCG@10,ServedRate
229,WL+Obs [ref],0.406667,0.773379,1.0
224,OAKG-lexicographic [ref],0.413333,0.737960,1.0
226,OAKG-similarity [ref],0.413333,0.737960,1.0
225,OAKG-product [ref],0.413333,0.737960,1.0
227,OAKG-threshold [ref],0.413333,0.737960,1.0
214,OAKG-product [ref],0.493333,0.579407,1.0
218,WL+Obs [ref],0.460000,0.566289,1.0
213,OAKG-lexicographic [ref],0.466667,0.544662,1.0
203,OAKG-product [ref],0.446667,0.544309,1.0
201,Missingness indicators [ref],0.426667,0.536270,1.0


## 7. What the real 512-case study found (the story)

The live demo is tiny; here are the **validated real numbers** (`results/`).

**7.1 The headline — OAKG beats masked cosine, everywhere.** OAKG's
support-restriction significantly beats masked cosine (the guidelines' key
baseline) in *every* masking regime and *every* missingness level: **+0.30
nDCG@10**, e.g. uniform +0.35, random +0.30, dataset-style +0.14, asymmetric +0.19.

**7.2 The honest nuance.** On *aggregate* random masking, OAKG only *ties* a
well-behaved zero-imputed cosine. So the story is **not** 'OAKG beats everything' —
it decisively beats comparison-restriction (masked cosine) and mixed-similarity
(Gower), and its distinctive value shows up on the hard cases and in semantics.

**7.3 Where OAKG separates from imputation — the hard-distractor stratum.**
When distractors are *designed* so that imputed zeros mislead (broad query, narrow
relevant case, broad distractor), OAKG separates significantly:

| Stratum | OAKG − zero-imputation | note |
|---|---|---|
| Hard-distractor (scaled) | **+0.068** [0.028, 0.115] SIG | 41 patient-level queries |
| Adversarial (best policy) | **+0.366** [0.220, 0.512] SIG | OAKG-similarity, perfect |
| Real-GT FLARE tumor | **+0.043** [0.019, 0.066] SIG | real cross-organ tumor labels |


## 8. The subtle finding that sharpened the method

On the adversarial stratum we learned **which part of OAKG does the work**:

- **Support-restriction (OAKG-*similarity*, no $\gamma$) = perfect** (+0.366).
- **$\gamma$-weighting (OAKG-*product*/*lexicographic*) *backfires*** — it
  down-weights *narrow-coverage* relevant cases (low $\gamma$), which are exactly
  the ones that should rank high.

> **So the primary policy is OAKG-similarity** (pure support-restriction);
> $\gamma$-product is retained only as an ablation that helps on the broad
> multi-organ benchmark. This is a genuine, honest refinement the experiments
> forced on us.


In [8]:
# Tiny illustration of the backfire: broad query (gamma=1 to another broad),
# narrow relevant (gamma small). product multiplies similarity by gamma ->
# penalizes the narrow relevant case even when its similarity S is high.
import numpy as np
for label, gamma, S in [('narrow-but-RELEVANT', 0.2, 0.95), ('broad-DISTRACTOR', 1.0, 0.60)]:
    print(f'{label:22} gamma={gamma:.2f} S={S:.2f} | similarity-policy R={S:.2f} | product-policy R={gamma*S:.2f}')
print('-> similarity-policy ranks the RELEVANT case first (0.95>0.60); product-policy wrongly ranks the DISTRACTOR first (0.60>0.19).')


narrow-but-RELEVANT    gamma=0.20 S=0.95 | similarity-policy R=0.95 | product-policy R=0.19
broad-DISTRACTOR       gamma=1.00 S=0.60 | similarity-policy R=0.60 | product-policy R=0.60
-> similarity-policy ranks the RELEVANT case first (0.95>0.60); product-policy wrongly ranks the DISTRACTOR first (0.60>0.19).


## 9. A qualitative win no retrieval metric captures — honest semantics

OAKG can answer a structured query with **three values**: True / False /
**Unknown**. On anatomy it never observed, it says *Unknown* — it does **not**
assert 'absent'. In the study this gives a **0.000 unsupported-negative rate**
(closed-world reasoning wrongly says 'absent' instead).


In [9]:
from oakg.semantics import evaluate_predicate
# a predicate about the pancreas, on a case where pancreas is masked out:
pred = {'feature':'pancreas_tumor_present','op':'==','value':1}
j = corpus.feature_index[pred['feature']]
# pick a case where the pancreas was NOT observed, so the predicate is unverifiable:
case = next((c for c in corpus.case_order if not Mm[corpus.case_to_row[c], j]), corpus.case_order[0])
print('case', case, '-> pancreas observed?', bool(Mm[corpus.case_to_row[case], j]))
for sem in ['closed_world','open_world','oakg']:
    v = evaluate_predicate(case, pred, Xm, Mm, sem, corpus)
    print(f'{sem:12} -> {v}   (T/F/U)')
print('OAKG/open-world return U (Unknown) when pancreas is unobserved; closed-world forces F (asserts absent) = an unsupported negative.')


case C0001 -> pancreas observed? False
closed_world -> F   (T/F/U)
open_world   -> U   (T/F/U)
oakg         -> U   (T/F/U)
OAKG/open-world return U (Unknown) when pancreas is unobserved; closed-world forces F (asserts absent) = an unsupported negative.


## 10. End-to-end recap

```
  segmentation masks (organs, tumor)
        |  oakg.phenotypes  (mask -> observations, with SUPPORT)
        v
  phenotype tables  +  MMKG schema (shared vocabulary)   <-- FLARE bridges datasets
        |  oakg.build_benchmark  (cases, phenotypes_ref/pred, queries,
        v                          organ-consistent relevance)
  observation masking (uniform/random/dataset-style/asymmetric)
        |  oakg.oakg  (support-restriction + gamma + policy)
        v
  per-query scores  ->  nDCG@10, paired bootstrap CIs  (oakg.pipeline)
        |
        v
  tables A-F + figures 1-2  +  strata (hard-distractor, real-GT tumor)
```

### The novel contribution, in three bullets
1. **Observability-aware comparison**: compare only observed, anatomically
   *supported* evidence — never impute-then-compare. Beats masked cosine in every
   regime and, on hard/tumor strata, beats imputation.
2. **A knowledge-graph substrate (MMKG)** that unifies heterogeneous single- and
   multi-organ datasets under one ontology, with FLARE as the multi-organ hub.
3. **Honest three-valued semantics**: OAKG abstains on unobserved anatomy
   (0 unsupported negatives) instead of fabricating absence.

### The considerations that made it rigorous (and honest)
- Organ-consistent relevance (cross-organ 'matches' are false positives).
- Support-restriction is the winning mechanism; $\gamma$-product *backfires* on
  narrow-relevant cases → primary policy is OAKG-similarity.
- Reference vs predicted tracks kept separate; predicted tumor rejected as
  unvalidatable → used **real** GT tumor (slice-level, clearly disclosed).
- Paired bootstrap CIs; results published to `results/` for the team.
